In [34]:
import numpy as np
import pandas as pd
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.manifold import MDS
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances
from scipy.spatial.distance import pdist, squareform


In [35]:
# =========================================================
# 0) READ DATA
# - rows: instance/locator
# - columns: type/feature
# - cell: abundance
# =========================================================

csv_path = "benthic_hellinger_num.csv"  # <-- path to your file
df = pd.read_csv(csv_path, sep=";")

##csv_path = "dune_hellinger.csv"
##df = pd.read_csv(csv_path, sep=",")

##csv_path = "mite_hellinger.csv"
##df = pd.read_csv(csv_path, sep=",")



In [36]:
#numeric_df = df.select_dtypes(include=[np.number])

In [37]:
# Get only the numeric columns
X = df.select_dtypes(include=[np.number]).copy()
X = X.fillna(0.0)

Xv = X.to_numpy(dtype=float)
n = Xv.shape[0]
if n < 3:
    raise ValueError("En az 3 örnek (satır) gerekli.")


In [38]:
# =========================================================
# 1) DISTANCE FUNCTIONS (pairwise)
# =========================================================
def bray_curtis_distance(x, y):
    num = np.sum(np.abs(x - y))
    den = np.sum(np.abs(x) + np.abs(y))
    return 0.0 if den == 0 else float(num / den)

def kulczynski_distance(x, y):
    sx = np.sum(x)
    sy = np.sum(y)
    m = np.sum(np.minimum(x, y))
    if sx == 0 and sy == 0:
        return 0.0
    if sx == 0 or sy == 0:
        return 1.0
    sim = 0.5 * (m / sx + m / sy)
    return float(1.0 - sim)

def chord_distance(x, y):
    nx = np.linalg.norm(x)
    ny = np.linalg.norm(y)
    if nx == 0 and ny == 0:
        return 0.0
    if nx == 0 or ny == 0:
        return 1.0
    x1 = x / nx
    y1 = y / ny
    return float(np.linalg.norm(x1 - y1))

# Chi-square distance
def chi_square_distance(x, y):
    # Oransal bolluk (pik)
    x_prop = x / np.sum(x) if np.sum(x) != 0 else np.zeros_like(x)
    y_prop = y / np.sum(y) if np.sum(y) != 0 else np.zeros_like(y)
    mean_prop = (x_prop + y_prop) / 2

    with np.errstate(divide='ignore', invalid='ignore'):
        dist = np.where(mean_prop == 0, 0, ((x_prop - y_prop) ** 2) / mean_prop)

    return np.sqrt(np.sum(dist))

# --- NB placeholder ---
def NB_distance(x, y):
    numerator = np.sum(np.abs(x - y) * (np.abs(x) + np.abs(y)))
    denominator = np.sum(x**2 + y**2)
    return 0.0 if denominator == 0 else float(numerator / denominator)

In [39]:
# -----------------------------
# 1) Dunn index 
# -----------------------------
def dunn_index(X, labels):
    distances = pairwise_distances(X)
    clusters = np.unique(labels)

    intra_dists = []
    for c in clusters:
        idx = np.where(labels == c)[0]
        if len(idx) > 1:
            intra_dists.append(np.max(distances[np.ix_(idx, idx)]))
        else:
            intra_dists.append(0.0)
    max_intra = np.max(intra_dists)

    inter_dists = []
    for i in range(len(clusters)):
        for j in range(i + 1, len(clusters)):
            idx_i = np.where(labels == clusters[i])[0]
            idx_j = np.where(labels == clusters[j])[0]
            inter_dists.append(np.min(distances[np.ix_(idx_i, idx_j)]))
    min_inter = np.min(inter_dists)

    return (min_inter / max_intra) if max_intra > 0 else 0.0


In [40]:
# -----------------------------
# 2) Assistants
# -----------------------------
def zscore_1d(a):
    a = np.asarray(a, dtype=float)
    mu = np.nanmean(a)
    sd = np.nanstd(a)
    if sd == 0 or np.isnan(sd):
        return np.zeros_like(a)
    return (a - mu) / sd


def compute_dist_matrix(X, distance_func, name=None):
    # Mahalanobis gibi özel durum eklemek isterseniz burada genişletin.
    return squareform(pdist(X, metric=distance_func))


def embedding_from_dist(dist_matrix, random_state=42, n_components=2):
    return MDS(
        n_components=n_components,
        dissimilarity="precomputed",
        random_state=random_state
    ).fit_transform(dist_matrix)


def metrics_for_k(coords, k, random_state=42):
    labels = KMeans(n_clusters=k, random_state=random_state).fit_predict(coords)
    sil = silhouette_score(coords, labels)
    db  = davies_bouldin_score(coords, labels)         # smaller good
    ch  = calinski_harabasz_score(coords, labels)      # larger good
    dn  = dunn_index(coords, labels)                   # larger good
    return sil, db, ch, dn


In [41]:
# -----------------------------
# 3) MAIN: Distance-agnostic common k selection (aggregate/median)
# - Extracts k=2..Kmax scores for each distance
# - Normalizes z-score within each distance (along k)
# - DB is reversed (smaller good -> larger good)
# - Gets MEDIAN aggregate score across distances for each k
# -----------------------------
def select_common_k_across_distances(
    X,
    distance_funcs: dict,
    k_range=range(2, 20),
    random_state=42,
    agg="median"   # "mean" or "median"
):
    rows = []
    coords_cache = {}

    #1) Extract embedding and metrics for each distance.
    for name, dist_fn in distance_funcs.items():
        dist_mat = compute_dist_matrix(X, dist_fn, name=name)
        coords = embedding_from_dist(dist_mat, random_state=random_state, n_components=2)
        coords_cache[name] = coords

        for k in k_range:
            # Ensure k is valid for silhouette_score
            # k must be < len(coords) for silhouette_score and other metrics
            if k >= len(coords) or k < 2:
                # print(f"Skipping k={k} for distance {name} because n_samples={len(coords)}")
                continue
            sil, db, ch, dn = metrics_for_k(coords, k, random_state=random_state)
            rows.append({
                "Distance": name,
                "k": int(k),
                "Silhouette": sil,
                "DaviesBouldin": db,
                "CalinskiHarabasz": ch,
                "Dunn": dn
            })

    long_df = pd.DataFrame(rows)

    #2) Normalize: within each distance, the z-score along k.
    long_df["Sil_z"]  = long_df.groupby("Distance")["Silhouette"].transform(zscore_1d)
    long_df["CH_z"]   = long_df.groupby("Distance")["CalinskiHarabasz"].transform(zscore_1d)
    long_df["Dunn_z"] = long_df.groupby("Distance")["Dunn"].transform(zscore_1d)

    # DB: Change sign and normalize so that smaller is good -> larger is good.
    long_df["DB_neg"] = -long_df["DaviesBouldin"]
    long_df["DB_z"]   = long_df.groupby("Distance")["DB_neg"].transform(zscore_1d)

    #3) Distance-based aggregate score (you can add weights if desired)
    long_df["Score_per_distance"] = long_df["Sil_z"] + long_df["CH_z"] + long_df["Dunn_z"] + long_df["DB_z"]

    #4) Aggregate along distances based on k
    if agg == "mean":
        k_table = long_df.groupby("k")["Score_per_distance"].mean().reset_index(name="AggregateScore")
    else:
        k_table = long_df.groupby("k")["Score_per_distance"].median().reset_index(name="AggregateScore")

    best_k = int(k_table.loc[k_table["AggregateScore"].idxmax(), "k"])

    return best_k, k_table, long_df, coords_cache


In [42]:
# -----------------------------
#4) Final table with selected common k
# -----------------------------
def final_results_with_common_k(coords_cache, common_k, random_state=42):
    rows = []
    for name, coords in coords_cache.items():
        labels = KMeans(n_clusters=common_k, random_state=random_state).fit_predict(coords)
        sil = silhouette_score(coords, labels)
        db  = davies_bouldin_score(coords, labels)
        ch  = calinski_harabasz_score(coords, labels)
        dn  = dunn_index(coords, labels)
        rows.append({
            "Distance": name,
            "k_used": common_k,
            "Silhouette": sil,
            "Davies-Bouldin": db,
            "Calinski-Harabasz": ch,
            "Dunn": dn
        })
    return pd.DataFrame(rows).sort_values("Silhouette", ascending=False)


In [43]:
# =========================================================
# USAGE
# =========================================================
# numeric_df should be ready:
#numeric_df = df.select_dtypes(include=[np.number])
#X = numeric_df.values

distance_funcs = {
    "NB": NB_distance,
    "Bray-Curtis": bray_curtis_distance,
    "Kulczynski": kulczynski_distance,
    "Chord": chord_distance,
    "Chi-square": chi_square_distance,
}

# Adjust k_range to ensure k is always < len(X) and <= 10 for sensibility
#k_range = range(2, min(len(X), 20))
k_range = range(2, 20)


In [44]:
common_k, k_table, long_df, coords_cache = select_common_k_across_distances(
    X=X,
    distance_funcs=distance_funcs,
    k_range=k_range,
    random_state=42,
    agg="median"   # "mean" 
)

print("Seçilen ORTAK k =", common_k)
display(k_table.sort_values("k"))

final_df = final_results_with_common_k(coords_cache, common_k, random_state=42)
display(final_df)


c:\Users\nursu\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\manifold\_mds.py:677: FutureWarning: The default value of `n_init` will change from 4 to 1 in 1.9.
  warnings.warn(
c:\Users\nursu\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\manifold\_mds.py:677: FutureWarning: The default value of `n_init` will change from 4 to 1 in 1.9.
  warnings.warn(
c:\Users\nursu\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\manifold\_mds.py:677: FutureWarning: The default value of `n_init` will change from 4 to 1 in 1.9.
  warnings.warn(
c:\Users\nursu\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\manifold\_mds.py:677: FutureWarning: The default value of `n_init` will change from 4 to 1 in 1.9.
  warnings.warn(
c:\Users\nursu\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\manifold\_mds.py:677: FutureWarning: The default value of `n_init` will change from 4 to 1 in 1.9.
  warnings.warn(


Seçilen ORTAK k = 11


,k,AggregateScore
0,2,-10.254890
1,3,-3.789741
2,4,-0.672820
3,5,-1.838874
4,6,-1.398806
5,7,1.358254
6,8,1.118483
7,9,1.480500
8,10,1.833244
9,11,2.396115


,Distance,k_used,Silhouette,Davies-Bouldin,Calinski-Harabasz,Dunn
3,Chord,11,0.507467,0.604287,216.546874,0.153383
2,Kulczynski,11,0.440756,0.748825,163.329363,0.151686
1,Bray-Curtis,11,0.433738,0.655106,196.309709,0.090305
4,Chi-square,11,0.422664,0.749467,193.210378,0.064164
0,NB,11,0.417742,0.725208,181.935428,0.152494
